In [4]:
import time
import os
import psutil
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, average_precision_score
from autogluon.tabular import TabularDataset, TabularPredictor

In [5]:
df = pd.read_csv("../../data/creditcard.csv")

In [6]:
# =========================
# COLUNAS
# =========================

X = df.drop(['Class'], axis=1)
y = df['Class'].astype(int)


In [7]:
# =========================
# FUNÇÃO AUXILIAR DE RECURSOS
# =========================
def get_memory_usage_mb():
    """Retorna o uso atual de memória RAM do processo em Megabytes."""
    process = psutil.Process(os.getpid())
    return round(process.memory_info().rss / (1024 * 1024), 2)

# =========================
# INSTALAR AUTOGLUON
# =========================
# %pip install autogluon psutil

for SEED in [42, 101, 223, 302, 404]:
    print(f"\n================================")
    print(f"EXECUTANDO COM SEED: {SEED}")
    print(f"================================")

    # =========================
    # DIVISÃO TREINO/TESTE
    # =========================
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=SEED, stratify=y
    )
    
    # =========================
    # DATAFRAME FINAL
    # =========================
    train_df = pd.DataFrame(X_train)
    train_df['Class'] = y_train.values

    test_df = pd.DataFrame(X_test)
    test_df['Class'] = y_test.values

    train_data = TabularDataset(train_df)
    test_data = TabularDataset(test_df)

    # =========================
    # AUTOGLUON (TREINAMENTO + MÉTRICA COMPUTACIONAL)
    # =========================
    print("Iniciando treinamento do AutoGluon...")
    mem_antes_treino = get_memory_usage_mb()
    inicio_treino = time.perf_counter()
    
    predictor = TabularPredictor(
        label='Class',
        eval_metric='f1' # Prioriza o F1-score da classe minoritária
    ).fit(
        train_data=train_data,
        presets='medium_quality',
        hyperparameters={
            'GBM': {'class_weight': 'balanced'}, # Força pesos balanceados no LightGBM
            'XGB': {'scale_pos_weight': 10}      # Penaliza erros na classe minoritária
        },
        time_limit=300
    )
    
    fim_treino = time.perf_counter()
    tempo_treino_total = round(fim_treino - inicio_treino, 2)
    mem_depois_treino = get_memory_usage_mb()
    memoria_treino_delta = round(mem_depois_treino - mem_antes_treino, 2)

    # =========================
    # LEADERBOARD
    # =========================
    print("\n===== LEADERBOARD =====")
    leaderboard = predictor.leaderboard(test_data, display=True)

    # =========================
    # MÉTRICAS DE TODOS OS MODELOS
    # =========================
    resultados = []

    X_test_ag = test_df.drop(columns=['Class'])
    y_true = test_df['Class'].values

    # Iterar sobre todos os modelos gerados
    for model_name in predictor.model_names():
        # Medição de tempo e uso de CPU/Memória na inferência específica do modelo
        inicio_inferencia = time.perf_counter()
        
        preds = predictor.predict(X_test_ag, model=model_name)
        y_pred = preds.astype(int).values
        
        y_score = predictor.predict_proba(X_test_ag, model=model_name, as_multiclass=False).values
        
        fim_inferencia = time.perf_counter()
        tempo_inferencia = round(fim_inferencia - inicio_inferencia, 4)

        # Métricas preditivas
        acc = round(accuracy_score(y_true, y_pred), 4)
        precision = round(precision_score(y_true, y_pred, pos_label=1, zero_division=0), 4)
        recall = round(recall_score(y_true, y_pred, pos_label=1, zero_division=0), 4)
        f1_weighted = round(f1_score(y_true, y_pred, pos_label=1, zero_division=0, average='weighted'), 4)
        f1 = round(f1_score(y_true, y_pred, pos_label=1, zero_division=0), 4)
        average_precision = round(average_precision_score(y_true, y_score), 4)

        resultados.append({
            "Modelo": model_name,
            "Tipo": model_name.split('_')[0],
            "Seed": SEED,
            "Accuracy": acc,
            "Precision": precision,
            "Recall": recall,
            "F1-score": f1,
            "F1-score Weighted": f1_weighted,
            "Average Precision": average_precision,
            "Tempo Treino Total (s)": tempo_treino_total,
            "Tempo Inferência (s)": tempo_inferencia,
            "Uso RAM Treino (MB)": mem_depois_treino
        })

    # =========================
    # RESULTADO FINAL
    # =========================
    df_resultados = pd.DataFrame(resultados)

    print("\n===== MÉTRiCAS E CUSTO COMPUTACIONAL - SEED {} =====".format(SEED))
    print(df_resultados.sort_values(by="Average Precision", ascending=False).to_string(index=False))

No path specified. Models will be saved in: "AutogluonModels/ag-20261106_152853"
Verbosity: 2 (Standard Logging)



EXECUTANDO COM SEED: 42
Iniciando treinamento do AutoGluon...


=================== System Info ===================
AutoGluon Version:  1.6.3
Python Version:     3.12.13
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.5.0: Mon Apr 27 20:39:29 PDT 2026; root:xnu-12377.121.6~2/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       3.92 GB / 16.00 GB (24.5%)
Disk Space Avail:   234.83 GB / 460.38 GB (51.0%)
Presets specified: ['medium_quality']
Beginning AutoGluon training ... Time limit = 300s
AutoGluon will save models to "/Users/lannaces/Documents/Inatel/TCC-autoML/notebooks/balanced/AutogluonModels/ag-20261106_152853"
Train Data Rows:    227845
Train Data Columns: 30
Label Column:       Class
AutoGluon infers your prediction problem is: 'binary' (because only two unique label-values observed).
	2 unique label values:  [np.int64(0)


===== LEADERBOARD =====
                 model  score_test  score_val eval_metric  pred_time_test  pred_time_val  fit_time  pred_time_test_marginal  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0              XGBoost    0.843750   0.857143          f1        0.030230       0.001946  0.707906                 0.030230                0.001946           0.707906            1       True          2
1             LightGBM    0.738318   0.857143          f1        0.003426       0.000490  0.453430                 0.003426                0.000490           0.453430            1       True          1
2  WeightedEnsemble_L2    0.738318   0.857143          f1        0.004238       0.001364  0.504347                 0.000812                0.000874           0.050917            2       True          3

===== MÉTRiCAS E CUSTO COMPUTACIONAL - SEED 42 =====
             Modelo             Tipo  Seed  Accuracy  Precision  Recall  F1-score  F1-score Weighted  Average Pre

No path specified. Models will be saved in: "AutogluonModels/ag-20261106_152857"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.3
Python Version:     3.12.13
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.5.0: Mon Apr 27 20:39:29 PDT 2026; root:xnu-12377.121.6~2/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       3.83 GB / 16.00 GB (23.9%)
Disk Space Avail:   234.78 GB / 460.38 GB (51.0%)
Presets specified: ['medium_quality']
Beginning AutoGluon training ... Time limit = 300s
AutoGluon will save models to "/Users/lannaces/Documents/Inatel/TCC-autoML/notebooks/balanced/AutogluonModels/ag-20261106_152857"
Train Data Rows:    227845
Train Data Columns: 30
Label Column:       Class
AutoGluon infers your predi

Iniciando treinamento do AutoGluon...


... (8 outputs left out)

No path specified. Models will be saved in: "AutogluonModels/ag-20261106_152903"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.3
Python Version:     3.12.13
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.5.0: Mon Apr 27 20:39:29 PDT 2026; root:xnu-12377.121.6~2/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       3.64 GB / 16.00 GB (22.8%)
Disk Space Avail:   234.60 GB / 460.38 GB (51.0%)
Presets specified: ['medium_quality']
Beginning AutoGluon training ... Time limit = 300s
AutoGluon will save models to "/Users/lannaces/Documents/Inatel/TCC-autoML/notebooks/balanced/AutogluonModels/ag-20261106_152903"
Train Data Rows:    227845
Train Data Columns: 30
Label Column:       Class
AutoGluon infers your predi

Iniciando treinamento do AutoGluon...


	Stage 4 Generators:
		Fitting DropUniqueFeatureGenerator...
	Stage 5 Generators:
		Fitting DropDuplicatesFeatureGenerator...
	Types of features in original data (raw dtype, special dtypes):
		('float', []) : 30 | ['Time', 'V1', 'V2', 'V3', 'V4', ...]
	Types of features in processed data (raw dtype, special dtypes):
		('float', []) : 30 | ['Time', 'V1', 'V2', 'V3', 'V4', ...]
	0.2s = Fit runtime
	30 features in original data used to generate 30 features in processed data.
	Train Data (Processed) Memory Usage: 52.15 MB (1.4% of available memory)
Data preprocessing and feature engineering runtime = 0.23s ...
AutoGluon will gauge predictive performance using evaluation metric: 'f1'
	To change this, specify the eval_metric parameter of Predictor()
Automatically generating train/validation split with holdout_frac=0.011, Train Rows: 225338, Val Rows: 2507
User-specified model hyperparameters to be fit:
{
	'GBM': [{'class_weight': 'balanced'}],
	'XGB': [{'scale_pos_weight': 10}],
}
Fitting 2 


===== LEADERBOARD =====
                 model  score_test  score_val eval_metric  pred_time_test  pred_time_val  fit_time  pred_time_test_marginal  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0             LightGBM    0.791667        1.0          f1        0.004088       0.000362  0.493017                 0.004088                0.000362           0.493017            1       True          1
1  WeightedEnsemble_L2    0.791667        1.0          f1        0.005048       0.001771  0.555447                 0.000960                0.001409           0.062430            2       True          3
2              XGBoost    0.757062        1.0          f1        0.031057       0.002263  0.736633                 0.031057                0.002263           0.736633            1       True          2

===== MÉTRiCAS E CUSTO COMPUTACIONAL - SEED 404 =====
             Modelo             Tipo  Seed  Accuracy  Precision  Recall  F1-score  F1-score Weighted  Average Pr